<a href="https://colab.research.google.com/github/zyuneee/portfolio/blob/main/deep%20learing/transfer_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content/'
!kaggle competitions download -c dogs-vs-cats-redux-kernels-edition
!unzip -q dogs-vs-cats-redux-kernels-edition.zip
!unzip -q train.zip -d .


 91% 737M/814M [00:04<00:02, 38.7MB/s]
100% 814M/814M [00:04<00:00, 196MB/s] 


In [6]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/dataset/',
    image_size=(150,150),
    batch_size=64,
    subset='training',
    validation_split = 0.2,
    seed=1234
)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.


In [7]:
val_ds= tf.keras.preprocessing.image_dataset_from_directory(
    '/content/dataset/',
    image_size=(150,150),
    batch_size=64,
    subset='validation',
    validation_split = 0.2,
    seed=1234

)

def 전처리함수(i,정답):
  i = tf.cast(i/255.0, tf.float32)
  return i, 정답

train_ds = train_ds.map(전처리함수)
val_ds = val_ds.map(전처리함수)

Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [17]:
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.applications import InceptionV3


In [ ]:
#InceptionV3.5는 InceptionV3라는모델의 가중치 값을 다 모아놓은 파일임
from tensorflow.keras.applications.inception_v3 import InceptionV3
#include_top=마지막출력층 가져올건지

inception_model = InceptionV3(input_shape=(150,150,3),include_top=False,weights=None)
inception_model.load_weights('inception_v3.h5')

inception_model.summary()

# 남의 레이어 다가져와놓고 그 안의 가중치 업데이트 시키면 안되니까 레이어 학습 금지시켜야함

for i in inception_model.layers:
  i.trainable = False #w값 고정  vs

#finetuning가능함
unfreeze = False
for i in inception_model.layers:
  if i.name == 'mixed6':
    unfreeze = True
    if unfreeze == True:
      i.trainable = True



마지막레이어 = inception_model.get.layer('mixed7') #원하는 레이어뽑기

In [ ]:
#내 레이어만들기

import tensorflow as tf

layer1 = tf.keras.layers.Flatten()(마지막레이어.output) #기존 모델레이어와, 내 레이어랑 연결
layer2 = tf.keras.layers.Dense(1024,activation='relu')(layer1)
drop1= tf.keras.layers.Dropout(0.2)(layer2)
layer3 = tf.keras.layers.Dense(1,activation='sigmoid')(drop1)

model = tf.keras.Model(inception_model.input,layer3)
model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['acc'])
mode.fit(train_ds, validation_data=val_ds, epochs=2)


#파인튜닝했을때 학습법
model.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(lr.=0.001),metrics=['acc'])
mode.fit(train_ds, validation_data=val_ds, epochs=2)